# Website Brochure Generator

This notebook scrapes a target website, uses an AI agent to identify
the most relevant pages, aggregates their content, and produces a
professional sales brochure in Markdown format.

## How it works

1. **Scrape** the landing page and extract all links
2. **Filter** links using an AI agent - only relevant pages proceed
3. **Aggregate** content from the selected pages
4. **Generate** a structured sales brochure via the OpenAI API
5. **Render** the brochure and provide a download link

In [ ]:
import os
import json
import requests

from pathlib import Path
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown, FileLink

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
BROCHURE_LANGUAGE = os.getenv("BROCHURE_LANGUAGE", "English")

if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. "
        "Copy .env.example to .env and add your key."
    )

client = OpenAI(api_key=OPENAI_API_KEY)

print(f"Model: {OPENAI_MODEL}")
print(f"Language: {BROCHURE_LANGUAGE}")
print("Client: Ready")

## Section 2 - Scraping Utilities

Functions responsible for fetching pages and extracting clean text
and links from raw HTML. All AI calls receive only sanitised content -
no raw HTML, no noise.

In [ ]:
# HTTP request headers — mimic a real browser to reduce bot-blocking
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

In [ ]:
def fetch_page(url: str) -> BeautifulSoup | None:
    """
    Fetch a URL and return a BeautifulSoup object.
    Returns None if the request fails or the content is not HTML.
    """

    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()

        # Skip non-HTML responses (PDFs, images, etc.)
        content_type = response.headers.get("Content-Type", "")

        if "text/html" not in content_type:
            print(f"Skipped non-HTML content: {url}")
            return None

        return BeautifulSoup(response.text, "html.parser")

    except requests.RequestException as e:
        print(f"Failed to fetch {url}: {e}")
        return None

In [ ]:
def extract_text(soup: BeautifulSoup) -> str:
    """
    Extract clean, readable text from a BeautifulSoup object.
    Removes scripts, styles and navigation noise before extraction.
    """

    # Remove elements that contribute no readable content
    for tag in soup(["script", "style", "nav", "footer", "form", "iframe"]):
        tag.decompose()

    text = soup.get_text(separator="\n", strip=True)
    lines = [line for line in text.splitlines() if line.strip()]

    return "\n".join(lines)

In [ ]:
def extract_links(soup: BeautifulSoup, base_url: str) -> list[dict]:
    """
    Extract all links from a page as a list of dicts.
    Each dict contains the absolute URL and the anchor text.
    Only includes http/https links - strips fragments and duplicates.
    """

    seen = set()
    links = []

    for tag in soup.find_all("a", href=True):
        href = tag["href"].strip()

        # Resolve relative URLs against the base URL
        if href.startswith("/"):
            parsed = urlparse(base_url)
            href = f"{parsed.scheme}://{parsed.netloc}{href}"

        # Skip non-HTTP/HTTPS links (mailto:, tel:, javascript:, etc.)
        if not href.startswith("http"):
            continue

        # Strip URL fragments (#section)
        href = href.split("#")[0].rstrip("/")

        if href not in seen:
            seen.add(href)

            links.append({
                "url": href,
                "text": tag.get_text(strip=True) or "(no anchor text)"
            })

    return links

## Section 3 - AI Link Filter

The AI agent receives the full list of links extracted from the landing page and decides which ones are worth visiting. It returns a structured JSON response - no guesswork, no regex, no hardcoded rules.

The agent is instructed to retain pages likely to contain information useful for a sales brochure: about, product, pricing, team, case studies, careers and similar. It discards noise: login pages, legal documents, blog tags, CDN assets and anything else that adds no value.

In [ ]:
LINK_FILTER_SYSTEM_PROMPT = f"""
You are a research assistant helping to build a sales brochure for a company.

You will receive a JSON array of links extracted from a company's website.
Each item contains a "url" and a "text" (the anchor text).

Your task is to select only the links that are likely to contain content useful for a professional sales brochure. Relevant pages include:
- About / mission / vision / values
- Products or services
- Pricing or plans
- Team or leadership
- Case studies or customer stories
- Careers (signals company culture and growth)
- Contact information

Discard all links that are irrelevant, such as:
- Login, signup or account pages
- Legal pages (privacy policy, terms of service, cookies)
- Blog tag or category pages (individual posts may be useful, tags are not)
- CDN assets, images or file downloads
- Social media profile links
- Duplicate or near-duplicate pages

Respond ONLY with a valid JSON object in this exact format:
{{
    "selected": [
        {{ "url": "https://example.com/about", "reason": "Contains company mission and values" }}
    ]
}}

Do not include any explanation, markdown or text outside the JSON object.
""".strip()

In [ ]:
def filter_links_with_agent(links: list[dict]) -> list[dict]:
    """
    Send the extracted links to the AI agent for filtering.
    Returns a list of selected link dicts, each with 'url' and 'reason'.

    Falls back to the full link list if the agent response cannot be parsed.
    """

    if not links:
        return []

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": LINK_FILTER_SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(links, ensure_ascii=False)}
        ],
        temperature=0.1,
        response_format={"type": "json_object"},
    )

    raw = response.choices[0].message.content.strip()

    try:
        parsed = json.loads(raw)
        selected = parsed.get("selected", [])

        print(f"Agent selected {len(selected)} links from {len(links)} total.")

        return selected

    except json.JSONDecodeError:
        print("Warning: agent returned malformed JSON. Using full link list.")
        return links

## Section 4 - Content Aggregation

Scrape each page selected by the AI agent and build a single content corpus. This corpus is passed directly to the brochure generation step.

Each page is fetched independently - failures are logged and skipped without interrupting the pipeline.

In [ ]:
MAX_PAGES = 15
MAX_CHARS_PER_PAGE = 5000


def aggregate_content(base_url: str, selected_links: list[dict]) -> str:
    """
    Scrape the base URL and all agent-selected pages.
    Returns a single aggregated string - the content corpus.

    Each page section is clearly delimited so the LLM understands which content belongs to which page.
    """

    pages_to_scrape = [{"url": base_url, "reason": "Landing page"}] + selected_links[:MAX_PAGES - 1]

    if len(selected_links) >= MAX_PAGES:
        print(f"Note: capped at {MAX_PAGES} pages (agent selected {len(selected_links) + 1}).")

    sections = []

    for page in pages_to_scrape:
        url = page["url"]
        reason = page.get("reason", "")

        print(f"Scraping: {url}")

        soup = fetch_page(url)

        if soup is None:
            print(f"    Skipped.")
            continue

        text = extract_text(soup)

        if not text.strip():
            print(f"    Empty content - skipped.")
            continue

        if len(text) > MAX_CHARS_PER_PAGE:
            text = text[:MAX_CHARS_PER_PAGE]
            print(f"    Truncated to {MAX_CHARS_PER_PAGE} characters.")

        section = (
            f"=== PAGE: {url} ===\n"
            f"=== REASON SELECTED: {reason}\n\n"
            f"{text}\n"
            f"=== END PAGE ===\n"
        )

        sections.append(section)

        print(f"    OK - {len(text)} characters extracted.")

    corpus = "\n\n".join(sections)

    print(f"\nCorpus ready - {len(sections)} pages, {len(corpus)} characters total.")

    return corpus

## Section 5 - Brochure Generation

The aggregated content corpus is passed to the model with a structured prompt. The model produces a professional sales brochure in Markdown format, written in the configured language.

In [ ]:
BROCHURE_SYSTEM_PROMPT = """
You are a professional copywriter specialising in B2B sales materials.

You will receive scraped content from multiple pages of a company website.
Your task is to write a polished, professional sales brochure in Markdown format.

The brochure must follow this structure:
1. Company name and a compelling one-line tagline.
2. About - who they are and what they stand for.
3. Products or Services - what they offer and key benefits.
4. Why Choose Us - differentiators, proof points, or customer outcomes.
5. Team or Leadership - if information is available.
6. Pricing or Plans - if information is available.
7. Call to Action - how to get in touch or get started.

Rules:
- Write in {language}.
- Be concise, confident, and persuasive. No filler sentences.
- Use clean Markdown: headings, bullet points, and bold for emphasis.
- Do not invent information. Use only what is present in the source content.
- Do not include emojis under any circumstances.
- If a section has no available data, omit it entirely - do not write placeholders.
""".strip()

In [ ]:
def generate_brochure(corpus: str, language: str = BROCHURE_LANGUAGE, stream: bool = True) -> str:
    """
    Generate a professional sales brochure from the content corpus.
    Returns the brochure as a Markdown string.

    When stream=True, tokens are printed live as they arrive.
    """

    system_prompt = BROCHURE_SYSTEM_PROMPT.format(language=language)

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": corpus},
        ],
        temperature=0.7,
        stream=stream,
    )

    if not stream:
        brochure = response.choices[0].message.content.strip()
        print(f"Brochure generated - {len(brochure)} characters.")
        return brochure

    chunks = []

    for chunk in response:
        delta = chunk.choices[0].delta.content

        if delta:
            print(delta, end="", flush=True)
            chunks.append(delta)

    print()  # newline after streaming finishes

    brochure = "".join(chunks).strip()

    print(f"\nBrochure generated - {len(brochure)} characters.")

    return brochure

## Section 6 - Output & Download

Run the full pipeline: scrape, filter, aggregate, generate.
The brochure is rendered below and saved as a Markdown file.

In [ ]:
TARGET_URL = "https://github.com/"  # replace with your target

print("=" * 60)
print("COMPANY BROCHURE GENERATOR")
print("=" * 60)
print(f"Target: {TARGET_URL}")
print(f"Model: {OPENAI_MODEL}")
print(f"Language: {BROCHURE_LANGUAGE}")
print("=" * 60)

# Step 1 — Scrape the landing page
print("\n[1/4] Scraping landing page...")

landing_soup = fetch_page(TARGET_URL)

if landing_soup is None:
    raise RuntimeError(f"Failed to fetch landing page: {TARGET_URL}.")

landing_links = extract_links(landing_soup, TARGET_URL)

print(f"    Found {len(landing_links)} links.")

# Step 2 — Filter links with the AI agent
print("\n[2/4] Filtering links with AI agent...")

selected_links = filter_links_with_agent(landing_links)

# Step 3 — Aggregate content
print("\n[3/4] Aggregating content...")

corpus = aggregate_content(TARGET_URL, selected_links)

# Step 4 — Generate brochure
print("\n[4/4] Generating brochure...")

brochure = generate_brochure(corpus)

print("\n" + "=" * 60)
print("Pipeline complete.")
print("=" * 60)

In [ ]:
# Save the brochure to a .md file
# Ensure the output directory exists before writing
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "brochure.md"
output_path.write_text(brochure, encoding="utf-8")

print(f"Brochure saved to: {output_path.resolve()}.")

# Render the brochure inline
display(Markdown("---"))
display(Markdown(brochure))
display(Markdown("---"))

# Provide a download link below the rendered brochure
display(FileLink(str(output_path), result_html_prefix="Download brochure: "))